In [1]:
from collections import defaultdict
import cv2
import numpy as np
from ultralytics import YOLO
import math
from shapely.geometry import LineString

model = YOLO('yolov8n.pt')

video_path = "traffic3.mp4"
cap = cv2.VideoCapture(video_path)

tl1 = (605, 80)
tr1 = (800, 80)
bl1 = (20, 700)
br1 = (1270, 700)

# Calculate the lengths of the two lines
length_line1 = math.sqrt((tr1[0] - tl1[0]) ** 2 + (tr1[1] - tl1[1]) ** 2)
length_line2 = math.sqrt((br1[0] - bl1[0]) ** 2 + (br1[1] - bl1[1]) ** 2)

first_frame_scale = 46080 / length_line1
car_width_scale1 = 8000 / first_frame_scale

print("first frame scale", first_frame_scale)
print("car_width_scale1", car_width_scale1)

track_history = defaultdict(lambda: [])
detected_vehicles = []

next_line_y3 = None

# Define the line y = 150
y_line = 150

# Function to find intersection point of two lines
def intersection(line1, line2):
    x_intersect, y_intersect = line1.intersection(line2).xy
    return x_intersect[0], y_intersect[0]


def create_line(frame, start_point, end_point, y_line):
    length = abs(end_point - start_point)
    line_scale = 46080 / length
    car_width_scale = 8000 / line_scale

    y = int(y_line + 1.5 * car_width_scale)
    line = LineString([(0, y), (frame.shape[1], y)])

    polygon_points = [tl1, tr1, br1, bl1, tl1]
    intersection_points = []

    for i in range(len(polygon_points) - 1):
        segment = LineString([polygon_points[i], polygon_points[i + 1]])
        if segment.intersects(line):
            x_intersect, _ = segment.intersection(line).xy
            intersection_points.append(x_intersect[0])

    intersection_points.sort()
    x_boundary_left = intersection_points[0]
    x_boundary_right = intersection_points[-1]

    return y, x_boundary_left, x_boundary_right

while cap.isOpened():
    success, frame = cap.read()
    if success:
        results = model.track(frame, persist=True)

        for result in results:
            if result.boxes is None or result.boxes.id is None:
                continue
            else:
                boxes = result.boxes.xywh.cpu()
                track_ids = result.boxes.id.cpu().numpy().astype(int)

                annotated_frame = result.plot()
                cv2.polylines(annotated_frame, [np.array([tl1, tr1, br1, bl1], dtype=np.int32)], True, (255, 0, 0),
                              2)

                for box, track_id in zip(boxes, track_ids):
                    x, y, w, h = box
                    track = track_history[track_id]
                    track.append((float(x), float(y)))
                    if len(track) > 30:
                        track.pop(0)

                    position = (int(x), int(y))
                    width = int(w)
                    detected_vehicles.append((track_id, position, width))

                    # Display track ID and width on the frame
                    cv2.putText(annotated_frame, f"ID: {track_id}", (position[0], position[1] - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
                    cv2.putText(annotated_frame, f"Width: {width}", (position[0], position[1] + 20),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

                line_y = int(tl1[1] + 1.5 * car_width_scale1)

                # Create LineString for the horizontal line
                line = LineString([(0, line_y), (frame.shape[1], line_y)])

                # Iterate over consecutive pairs of points defining the polygon
                polygon_points = [tl1, tr1, br1, bl1, tl1]  # Add the first point at the end to complete the loop
                intersection_points = []
                for i in range(len(polygon_points) - 1):
                    # Create LineString for each segment of the polygon
                    segment = LineString([polygon_points[i], polygon_points[i + 1]])

                    # Check if the segment intersects with the horizontal line
                    if segment.intersects(line):
                        # If there's an intersection, calculate the intersection point
                        x_intersect, _ = intersection(segment, line)

                        # Add the x-coordinate of the intersection point to the list
                        intersection_points.append(x_intersect)

                # Sort the intersection points to get the boundaries
                intersection_points.sort()

                # The boundaries are the first and last intersection points
                x_boundary_left = intersection_points[0]
                x_boundary_right = intersection_points[-1]

#                 print("x-coordinate boundaries:", x_boundary_left, x_boundary_right)
                
                cv2.line(annotated_frame, (int(x_boundary_left), line_y), (int(x_boundary_right), line_y), (0, 255, 0), 2)
                
                line_y2, x_boundary_left2, x_boundary_right2 = create_line(frame, x_boundary_left, x_boundary_right, line_y)
                
                cv2.line(annotated_frame, (int(x_boundary_left2), line_y2), (int(x_boundary_right2), line_y2), (0, 255, 0), 2)
                
                print("sector 3 boundaries:", (int(x_boundary_left2), line_y2), (int(x_boundary_right2), line_y2))
                
                line_y3, x_boundary_left3, x_boundary_right3 = create_line(frame, x_boundary_left2, x_boundary_right2, line_y2)
                
                cv2.line(annotated_frame, (int(x_boundary_left3), line_y3), (int(x_boundary_right3), line_y3), (0, 255, 0), 2)
                
                print("sector 3 boundaries:", (int(x_boundary_left3), line_y3), (int(x_boundary_right3), line_y3))
                
                line_y4, x_boundary_left4, x_boundary_right4 = create_line(frame, x_boundary_left3, x_boundary_right3, line_y3)
                
                cv2.line(annotated_frame, (int(x_boundary_left4), line_y4), (int(x_boundary_right4), line_y4), (0, 255, 0), 2)
                
                line_y5, x_boundary_left5, x_boundary_right5 = create_line(frame, x_boundary_left4, x_boundary_right4, line_y4)
                
                cv2.line(annotated_frame, (int(x_boundary_left5), line_y5), (int(x_boundary_right5), line_y5), (0, 255, 0), 2)
                
                                
        cv2.imshow("YOLOv8 Tracking", annotated_frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
        if next_line_y3 is not None:
            break
    else:
        break

cap.release()
cv2.destroyAllWindows()


first frame scale 236.30769230769232
car_width_scale1 33.854166666666664

0: 384x640 1 person, 7 cars, 513.4ms
Speed: 10.5ms preprocess, 513.4ms inference, 36.4ms postprocess per image at shape (1, 3, 384, 640)


[W NNPACK.cpp:64] Could not initialize NNPACK! Reason: Unsupported hardware.


sector 3 boundaries: (489, 202) (892, 202)
sector 3 boundaries: (391, 306) (971, 306)

0: 384x640 1 person, 7 cars, 139.0ms
Speed: 1.8ms preprocess, 139.0ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)
sector 3 boundaries: (489, 202) (892, 202)
sector 3 boundaries: (391, 306) (971, 306)

0: 384x640 1 person, 7 cars, 187.7ms
Speed: 2.1ms preprocess, 187.7ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)
sector 3 boundaries: (489, 202) (892, 202)
sector 3 boundaries: (391, 306) (971, 306)

0: 384x640 1 person, 7 cars, 181.9ms
Speed: 3.4ms preprocess, 181.9ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)
sector 3 boundaries: (489, 202) (892, 202)
sector 3 boundaries: (391, 306) (971, 306)



2025-01-12 04:04:32.556 python[45200:3218648] +[IMKClient subclass]: chose IMKClient_Legacy
2025-01-12 04:04:32.556 python[45200:3218648] +[IMKInputSession subclass]: chose IMKInputSession_Legacy


0: 384x640 1 person, 7 cars, 192.4ms
Speed: 2.7ms preprocess, 192.4ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)
sector 3 boundaries: (489, 202) (892, 202)
sector 3 boundaries: (391, 306) (971, 306)

0: 384x640 1 person, 8 cars, 180.7ms
Speed: 3.2ms preprocess, 180.7ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)
sector 3 boundaries: (489, 202) (892, 202)
sector 3 boundaries: (391, 306) (971, 306)

0: 384x640 1 person, 8 cars, 155.1ms
Speed: 2.3ms preprocess, 155.1ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)
sector 3 boundaries: (489, 202) (892, 202)
sector 3 boundaries: (391, 306) (971, 306)

0: 384x640 1 person, 8 cars, 196.9ms
Speed: 3.0ms preprocess, 196.9ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)
sector 3 boundaries: (489, 202) (892, 202)
sector 3 boundaries: (391, 306) (971, 306)

0: 384x640 1 person, 8 cars, 158.6ms
Speed: 2.3ms preprocess, 158.6ms inference, 1.8ms postprocess per image at 

sector 3 boundaries: (489, 202) (892, 202)
sector 3 boundaries: (391, 306) (971, 306)

0: 384x640 8 cars, 228.4ms
Speed: 2.6ms preprocess, 228.4ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)
sector 3 boundaries: (489, 202) (892, 202)
sector 3 boundaries: (391, 306) (971, 306)

0: 384x640 8 cars, 225.2ms
Speed: 2.4ms preprocess, 225.2ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)
sector 3 boundaries: (489, 202) (892, 202)
sector 3 boundaries: (391, 306) (971, 306)

0: 384x640 8 cars, 152.8ms
Speed: 2.4ms preprocess, 152.8ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)
sector 3 boundaries: (489, 202) (892, 202)
sector 3 boundaries: (391, 306) (971, 306)

0: 384x640 8 cars, 196.6ms
Speed: 2.4ms preprocess, 196.6ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)
sector 3 boundaries: (489, 202) (892, 202)
sector 3 boundaries: (391, 306) (971, 306)

0: 384x640 9 cars, 158.2ms
Speed: 2.5ms preprocess, 158.2ms infer

sector 3 boundaries: (489, 202) (892, 202)
sector 3 boundaries: (391, 306) (971, 306)

0: 384x640 8 cars, 181.4ms
Speed: 2.4ms preprocess, 181.4ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)
sector 3 boundaries: (489, 202) (892, 202)
sector 3 boundaries: (391, 306) (971, 306)

0: 384x640 8 cars, 174.3ms
Speed: 2.4ms preprocess, 174.3ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)
sector 3 boundaries: (489, 202) (892, 202)
sector 3 boundaries: (391, 306) (971, 306)

0: 384x640 8 cars, 162.5ms
Speed: 3.8ms preprocess, 162.5ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)
sector 3 boundaries: (489, 202) (892, 202)
sector 3 boundaries: (391, 306) (971, 306)

0: 384x640 8 cars, 184.6ms
Speed: 3.0ms preprocess, 184.6ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)
sector 3 boundaries: (489, 202) (892, 202)
sector 3 boundaries: (391, 306) (971, 306)

0: 384x640 8 cars, 194.3ms
Speed: 2.6ms preprocess, 194.3ms infer

0: 384x640 9 cars, 170.5ms
Speed: 2.5ms preprocess, 170.5ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)
sector 3 boundaries: (489, 202) (892, 202)
sector 3 boundaries: (391, 306) (971, 306)

0: 384x640 10 cars, 175.6ms
Speed: 2.4ms preprocess, 175.6ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)
sector 3 boundaries: (489, 202) (892, 202)
sector 3 boundaries: (391, 306) (971, 306)

0: 384x640 10 cars, 162.7ms
Speed: 2.6ms preprocess, 162.7ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)
sector 3 boundaries: (489, 202) (892, 202)
sector 3 boundaries: (391, 306) (971, 306)

0: 384x640 10 cars, 186.3ms
Speed: 2.4ms preprocess, 186.3ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)
sector 3 boundaries: (489, 202) (892, 202)
sector 3 boundaries: (391, 306) (971, 306)

0: 384x640 10 cars, 177.0ms
Speed: 2.4ms preprocess, 177.0ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)
sector 3 boundaries: (4

sector 3 boundaries: (489, 202) (892, 202)
sector 3 boundaries: (391, 306) (971, 306)

0: 384x640 10 cars, 170.0ms
Speed: 2.3ms preprocess, 170.0ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)
sector 3 boundaries: (489, 202) (892, 202)
sector 3 boundaries: (391, 306) (971, 306)

0: 384x640 10 cars, 180.3ms
Speed: 2.6ms preprocess, 180.3ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)
sector 3 boundaries: (489, 202) (892, 202)
sector 3 boundaries: (391, 306) (971, 306)

0: 384x640 10 cars, 163.2ms
Speed: 2.3ms preprocess, 163.2ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)
sector 3 boundaries: (489, 202) (892, 202)
sector 3 boundaries: (391, 306) (971, 306)

0: 384x640 10 cars, 176.8ms
Speed: 2.3ms preprocess, 176.8ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)
sector 3 boundaries: (489, 202) (892, 202)
sector 3 boundaries: (391, 306) (971, 306)

0: 384x640 10 cars, 164.5ms
Speed: 2.5ms preprocess, 164.5ms 

## 